In [1]:

# analysis_utils.py
# Unified Feature Extraction Module – FINAL VERSION (6 Features)
# LLM Comparative Evaluation Framework – Week 4 Deliverable
# Prepared by: Negin Latifi | 


In [2]:
from typing import Any, Dict
import re
import math
import warnings


# =========================== Shared Helper ===========================
def _is_nan(value: Any) -> bool:
    """Robust None/NaN detector used by all functions."""
    if value is None:
        return True
    if isinstance(value, float):
        return math.isnan(value)
    return False


In [3]:

# =========================== 1. Writing Style Analysis ===========================
def analyzeWritingStyle(text: Any) -> Dict[str, Any]:
    if _is_nan(text) or not str(text).strip():
        return {
            "is_detailed": False,
            "has_step_by_step": False,
            "word_count": 0,
            "sentence_count": 0,
            "avg_words_per_sentence": 0.0,
        }

    s = str(text)
    words = s.split()
    word_count = len(words)
    sentences = [seg.strip() for seg in re.split(r"[.!?]+\s*", s) if seg.strip()]
    sentence_count = len(sentences) if sentences else 1
    avg_words_per_sentence = word_count / sentence_count

    step_patterns = [
        r"\bstep\s*\d+", r"\bpart\s*\d+", r"\bsection\s*\d+", r"\bphase\s*\d+", r"\bstage\s*\d+",
        r"\bfirst\b", r"\bsecond\b", r"\bthird\b", r"\bfourth\b", r"\bfifth\b",
        r"\binitially\b", r"\bsubsequently\b", r"\bnext\b", r"\bthen\b", r"\bafter that\b", r"\bfinally\b",
        r"\btherefore\b", r"\bthus\b", r"\bconsequently\b", r"\bas a result\b",
        r"\bthe following steps\b", r"\bto break this down\b", r"\bwe proceed as follows\b"
    ]
    has_step_by_step = any(re.search(p, s.lower()) for p in step_patterns)
    is_detailed = (avg_words_per_sentence > 15) and (word_count > 50)

    return {
        "is_detailed": is_detailed,
        "has_step_by_step": has_step_by_step,
        "word_count": word_count,
        "sentence_count": sentence_count,
        "avg_words_per_sentence": round(avg_words_per_sentence, 2),
    }

In [4]:
# =========================== 2. Language Detection (Core Helper) ===========================
def _detect_language_group(s: str) -> str:
    """Priority order: Persian/Arabic → Chinese → Latin → Other"""
    if re.search(r"[\u0600-\u06FF]", s):      # Persian & Arabic first
        return "persian_arabic"
    if re.search(r"[\u4E00-\u9FFF]", s):      # Chinese second
        return "chinese"
    if re.search(r"[A-Za-z]", s):             # Latin last
        return "latin"
    return "other"


In [5]:
# =========================== 3. Token Counting – Language-Adaptive ===========================
def countTokens(text: Any) -> int:
    if _is_nan(text) or not str(text).strip():
        return 0
    s = re.sub(r"\s+", " ", str(text).strip())
    if not s:
        return 0

    lang = _detect_language_group(s)
    ratios = {"latin": 3.8, "persian_arabic": 2.4, "chinese": 1.2}
    ratio = ratios.get(lang, 4.0)
    return max(1, round(len(s) / ratio))


In [6]:

# =========================== 4. Code Block Detection ===========================
def detectCodeBlock(text: Any) -> bool:
    if _is_nan(text) or not str(text).strip():
        return False
    s = str(text)
    patterns = [
        r"```[\s\S]*?```", r"```[^`]+```", r"^ {4}.+", r"^\t.+",
        r"`[^`\n]+`", r"<code>[\s\S]*?</code>", r"<pre>[\s\S]*?</pre>",
        r"\bdef\s+\w+\(", r"\bclass\s+\w+", r"\bfunction\s+\w+",
        r"#include\s+<", r"\bfor\s*\(", r"\bwhile\s*\("
    ]
    return any(re.search(p, s, re.MULTILINE | re.IGNORECASE) for p in patterns)


In [7]:
# =========================== 5. Emoji Detection ===========================
_EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"   # emoticons
    "\U0001F300-\U0001F5FF"   # symbols & pictographs
    "\U0001F680-\U0001F6FF"   # transport & map symbols
    "\U0001F1E0-\U0001F1FF"   # flags
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE
)

def detectEmojis(text: Any) -> bool:
    if _is_nan(text) or not str(text).strip():
        return False
    return bool(_EMOJI_PATTERN.search(str(text)))

In [8]:

# =========================== 6. Table Detection ===========================
def detectTables(text: Any) -> bool:
    if _is_nan(text) or not str(text).strip():
        return False
    s = str(text)
    return bool(
        re.search(r"\|.*\|.*\|", s, re.MULTILINE) or
        re.search(r"<table|<tr|<td|<th", s, re.IGNORECASE)
    )



In [10]:

# =========================== 7. Language Detection – Public Function (Bonus Feature!) ===========================
def detectLanguage(text: Any) -> str:
    """
    Detects the dominant language group of the text.
    Returns one of: "persian_arabic", "chinese", "latin", "other", "unknown"
    """
    if _is_nan(text) or not str(text).strip():
        return "unknown"
    return _detect_language_group(str(text))